In [3]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader as TorchDataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import EdgeConv, SAGEConv, global_mean_pool, global_max_pool

In [4]:
graphs = torch.load("datasets/qg_graph_dataset_k8_edgeattr.pt", weights_only=False)
y = []
for g in graphs:
    y.append(g.y.item())
y = np.array(y)

In [37]:
TEMPERATURE = 0.5
PROJ_DIM = 128
HIDDEN_DIM = 64

# set division
VAL_SIZE = 0.15
TEST_SIZE = 0.15

EPOCHS = 20
BASELINE_EPOCHS = 8
BATCH_SIZE = 256

LR = 1e-4
WEIGHT_DECAY = 1e-4

DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [41]:
#stratified split
train_idx, temp_idx = train_test_split(np.arange(len(graphs)), test_size=VAL_SIZE + TEST_SIZE, stratify=y)

val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y[temp_idx])

In [42]:
subset_idx = train_test_split(
    np.arange(len(graphs)),
    train_size=20000,
    stratify=y,
    random_state=42
)[0]

subset_graphs = [graphs[i] for i in subset_idx]
subset_y = y[subset_idx]

In [43]:
train_graphs = [graphs[i] for i in train_idx]
val_graphs = [graphs[i] for i in val_idx]
test_graphs = [graphs[i] for i in test_idx]

In [44]:
val_loader  = PyGDataLoader(val_graphs,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = PyGDataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [45]:
class SimCLRGraphDataset(Dataset):
    def __init__(self, graphs, p_node=0.1, p_edge=0.1):
        self.graphs = graphs
        self.p_node = p_node
        self.p_edge = p_edge
    
    def __len__(self):
        return len(self.graphs)
    
    def __getitem__(self, idx):
        data =self.graphs[idx]
        view1 = self.augment(data)
        view2 = self.augment(data)
        return view1, view2
    
    def augment(self, data):
        data = self.augment_node_features(data)
        data = self.augment_edges(data)
        return data

    def augment_node_features(self, data):
        data_copy = data.clone()
        mask = torch.bernoulli(torch.full((data_copy.num_nodes, 1), 1 - self.p_node))
        data_copy.x = data_copy.x * mask
        return data_copy

    def augment_edges(self, data):
        data_copy = data.clone()
        mask = torch.bernoulli(torch.full((data_copy.num_edges, ), 1 - self.p_edge))
        data_copy.edge_index = data_copy.edge_index[:, mask.bool()]
        data_copy.edge_attr = data_copy.edge_attr[mask.bool()]
        return data_copy

In [46]:
sub_train_idx, sub_temp_idx = train_test_split(
    np.arange(len(subset_graphs)),
    test_size=VAL_SIZE + TEST_SIZE,
    stratify=subset_y,
    random_state=42
)
sub_val_idx, sub_test_idx = train_test_split(
    sub_temp_idx,
    test_size=0.5,
    stratify=subset_y[sub_temp_idx],
    random_state=42
)

sub_train_graphs = [subset_graphs[i] for i in sub_train_idx]
sub_val_graphs   = [subset_graphs[i] for i in sub_val_idx]
sub_test_graphs  = [subset_graphs[i] for i in sub_test_idx]

train_dataset = SimCLRGraphDataset(sub_train_graphs)
val_dataset   = SimCLRGraphDataset(sub_val_graphs)

In [47]:
from torch_geometric.data import Batch

def collate_fn(batch):
    batch_view1, batch_view2 = zip(*batch)
    batch_view1 = list(batch_view1)
    batch_view2 = list(batch_view2)
    batch_view1 = Batch.from_data_list(batch_view1)
    batch_view2 = Batch.from_data_list(batch_view2)
    return batch_view1, batch_view2

In [48]:
train_loader = PyGDataLoader(train_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [49]:
batch_view1, batch_view2 = next(iter(train_loader))
print(batch_view1)
print(batch_view2)

DataBatch(x=[179079, 6], edge_index=[2, 1376477], edge_attr=[1376477, 4], y=[256], pos=[179079, 2], batch=[179079], ptr=[257])
DataBatch(x=[179079, 6], edge_index=[2, 1375984], edge_attr=[1375984, 4], y=[256], pos=[179079, 2], batch=[179079], ptr=[257])


In [50]:
class GNNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels = 64):
        super().__init__()

        self.conv1 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * in_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU()
            )
        )

        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU()
            )
        )

        self.bn2 = nn.BatchNorm1d(hidden_channels)
    
    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)

        x_mean_pool = global_mean_pool(x, batch)
        x_max_pool = global_max_pool(x, batch)

        x = torch.cat([x_mean_pool, x_max_pool], dim=1)

        return x
    

In [51]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, x):
        x = self.mlp(x)
        return x

In [52]:
class SimCLRModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, proj_dim):
        super().__init__()
        self.encoder = GNNEncoder(in_channels, hidden_channels)
        self.projector = ProjectionHead(2 * hidden_channels, proj_dim)

    def forward(self, x, edge_index, batch):
        x = self.encoder(x, edge_index, batch)
        x = self.projector(x)
        x_norm = F.normalize(x, dim=1)
        return x_norm

In [53]:
in_channels = graphs[0].x.shape[1]
CLR_model = SimCLRModel(in_channels=in_channels, hidden_channels=HIDDEN_DIM, proj_dim=PROJ_DIM).to(DEVICE)

In [54]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
        sim_matrix = torch.cat([z1, z2], dim=0)
        sim_matrix = (sim_matrix @ sim_matrix.T / self.temperature)
        mask = torch.eye(sim_matrix.shape[0]).to(z1.device)
        mask = mask.to(torch.bool)
        sim_matrix = sim_matrix.masked_fill(mask, float('-inf'))

        batch_size = z1.shape[0]
        pos_labels = torch.cat([torch.arange(batch_size, 2 * batch_size),torch.arange(0, batch_size)], dim=0)

        loss = F.cross_entropy(sim_matrix, pos_labels.to(z1.device))
        return loss

In [55]:
CLR_optimizer = torch.optim.Adam(CLR_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
CLR_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(CLR_optimizer, mode='max')
CLR_criterion = NTXentLoss(temperature=TEMPERATURE)


In [56]:
def train_one_epoch_CLR(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    total_graphs = 0
    
    for batch in tqdm(loader, desc=f"Training", leave=False):
        view1, view2 = batch

        view1 = view1.to(DEVICE)
        view2 = view2.to(DEVICE)

        optimizer.zero_grad()
        z1 = model(view1.x, view1.edge_index, view1.batch)
        z2 = model(view2.x, view2.edge_index, view2.batch)
        loss = criterion(z1,z2)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        total_loss += loss.item() * view1.num_graphs
        total_graphs += view1.num_graphs

    return total_loss / total_graphs

In [57]:
@torch.no_grad()
def evaluate_CLR(model, loader, criterion):
    model.eval()

    total_loss = 0
    total_graphs = 0

    for view1, view2 in loader:
        view1 = view1.to(DEVICE)
        view2 = view2.to(DEVICE)

        z1 = model(view1.x, view1.edge_index, view1.batch)
        z2 = model(view2.x, view2.edge_index, view2.batch)
        loss = criterion(z1,z2)

        total_loss += loss.item() * view1.num_graphs
        total_graphs += view1.num_graphs
    return total_loss / total_graphs

In [58]:
val_dataset = SimCLRGraphDataset(val_graphs)
val_loader_CLR = PyGDataLoader(val_dataset, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [59]:
best_val_loss = float("inf")
patience = 3
patience_counter = 0
best_model_path = "models/best_CLR.pt"

In [61]:
history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch_CLR(CLR_model, train_loader, CLR_optimizer, CLR_criterion)
    val_loss = evaluate_CLR(CLR_model, val_loader_CLR, CLR_criterion)
    CLR_scheduler.step(val_loss)

    history.append({
        'epoch': epoch,
        'val_loss': val_loss,
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(CLR_model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

Epoch 01/20 | train_loss=4.9270 | val_loss=4.8210


Epoch 02/20 | train_loss=4.7416 | val_loss=4.7085


Epoch 03/20 | train_loss=4.6659 | val_loss=4.6550


Epoch 04/20 | train_loss=4.6199 | val_loss=4.6216


Epoch 05/20 | train_loss=4.5831 | val_loss=4.5803


...

Epoch 16/20 | train_loss=4.4056 | val_loss=4.4173


Epoch 17/20 | train_loss=4.4053 | val_loss=4.4147


Epoch 18/20 | train_loss=4.4042 | val_loss=4.4158


Epoch 19/20 | train_loss=4.4031 | val_loss=4.4128


Epoch 20/20 | train_loss=4.4027 | val_loss=4.4117


In [62]:
CLR_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

# congela os pesos do encoder
for param in CLR_model.encoder.parameters():
    param.requires_grad = False